[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [scikit-learn, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)

# Cross-Validation


## What you will be able to do

Score a model on every station by turns with cross-validation, choose and shuffle the folds, and
report the spread of the fold scores alongside their mean.

## The idea

### The problem

A single split gives a single score, and the score depends on which stations happened to land in the
test quarter. Split the 1,200 stations with four different seeds and the same linear model scores
0.888, 0.875, 0.876 and 0.873. None of those is wrong, and none of them is the model's score. Each
is the score on one particular 300 stations.

The test set has a second limit. Every time you look at it to compare two models or two settings,
your choices start to fit those 300 stations, and the score you finally report drifts upward from
what new stations will show. A test set used once at the end is still the honest last check, as the
**Train and Test** notebook argued. For the comparisons along the way, you need scores that come
from the training stations alone.

### What cross-validation is

> **k-fold cross-validation** splits the rows into k parts, called **folds**, then fits the model k
> times, each time on every fold but one, and scores it on the fold it left out. Every row is scored
> once by a model that never saw it, and the k scores, their **mean** and their **spread**, describe
> how the model does on rows it has not seen.

### Why it works that way

Each fit trains on most of the rows, four fifths of them with five folds, so each fold's score is
close to what a model trained on all of them would get, and every station is tested exactly once.
The mean uses all 1,200 tests instead of 300. The spread is information too. A model whose fold
scores differ widely depends heavily on which rows it gets, and a difference between two models
smaller than that spread may be luck.

Two details decide what the folds hold. Given a whole number, `cv=5`, scikit-learn splits the rows
in the order they come, without shuffling, into `KFold` folds for a regressor and `StratifiedKFold`
folds for a classifier. A table that arrives sorted then gives each fold one band of it. Pass a
splitter with `shuffle=True` and a `random_state` instead, and the folds are mixed but the same on
every run. **Stratified** folds keep the share of each class the same in every fold, which matters
most when one class is rare.

Scores follow one rule: larger is better. So an error, where smaller is better, is reported negated,
as in `"neg_mean_absolute_error"`, and a mean absolute error of 0.8 degrees comes back as -0.8.

### Where this shows up

The **Pipelines** notebook scored whole pipelines by folds, so that every step was fitted again for
each one. The **Hyperparameter Search** notebook runs cross-validation inside its search, once for
every setting it tries, and the **Imbalanced Classes** notebook depends on stratified folds. The
statistics of why the fold mean estimates performance are in the Visual Handbook.

### What this notebook covers

- one split against five folds, and the spread of the fold scores
- the folds themselves, shuffled and seeded
- two models compared on the same folds
- stratified folds for a rare class, and training scores from `cross_validate`
- three errors, among them a sorted table that makes four of five folds score below zero

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things.

```python
from sklearn.model_selection import KFold

stations = ["A", "B", "C", "D", "E", "F"]
for train, test in KFold(3).split(stations):
    print("train on", [stations[i] for i in train], "| test on", [stations[i] for i in test])
```

```
train on ['C', 'D', 'E', 'F'] | test on ['A', 'B']
train on ['A', 'B', 'E', 'F'] | test on ['C', 'D']
train on ['A', 'B', 'C', 'D'] | test on ['E', 'F']
```

Three folds of two stations each. Every station is in exactly one test fold, and each model trains
on the other four. Without shuffling, the folds follow the order of the list.

## Setup

Two libraries for the data, scikit-learn, and the models and splitters this notebook compares.

- `numpy` and `pandas` build the stations, and the cell prints scikit-learn's version
- `LinearRegression` and `RandomForestRegressor` are the two models of January temperature
- `LogisticRegression`, scaled by `StandardScaler` in a pipeline from `make_pipeline`, predicts snow
  on 1 May
- `train_test_split` makes the single splits the worked examples start from
- `cross_val_score` and `cross_validate` score a model by folds, and `KFold` and `StratifiedKFold`
  make the folds
- `warnings` catches the warning in the last Common error

`station_network` makes a network of 1,200 weather stations in Norway. The stations are made up, and
a generator started from a fixed seed draws them, so the table is the same on every run. Each row
has a station's latitude, longitude, elevation in meters, distance from the coast in kilometers and
the year it was installed, its terrain and owner as text, and three things to predict: its January
mean in degrees Celsius, whether it has snow on 1 May, and whether it has a frost in June.
`FEATURES` names the five numeric columns the models use.

In [1]:
import warnings

import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import (KFold, StratifiedKFold, cross_val_score, cross_validate,
                                     train_test_split)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def station_network(n=1200, seed=2025):
    """A made-up network of weather stations in Norway, one row each, with three things to predict."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    terrain = np.where(coast_km < 3, "coastal",
              np.where(elevation > 700, "mountain",
              np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), "fjord", "inland")))
    owner = rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2])
    installed = rng.integers(1950, 2025, n)
    shift = pd.Series(terrain).map({"coastal": 0.8, "fjord": 0.3, "inland": 0.0, "mountain": -0.5}).to_numpy()
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    june_min = 8.5 - 0.35 * (latitude - 58) - 0.0075 * elevation - 0.02 * coast_km + rng.normal(0, 1.5, n)
    table = pd.DataFrame({
        "station": [f"ST{i:04d}" for i in range(1, n + 1)],
        "latitude": latitude.round(2), "longitude": longitude.round(2),
        "elevation_m": elevation.round(0).astype(int), "coast_km": coast_km.round(1),
        "terrain": terrain, "owner": owner, "installed": installed,
        "jan_mean": jan_mean.round(1),
        "snow_on_may_1": may_mean < 2.5,
        "june_frost": june_min < -0.8,
    })
    return table


network = station_network()
FEATURES = ["latitude", "longitude", "elevation_m", "coast_km", "installed"]

print("scikit-learn", sklearn.__version__, "|", len(network), "stations")

scikit-learn 1.6.1 | 1200 stations


## Worked examples

### Scores from different splits

The **Train and Test** notebook split the stations once. Here is the same linear model scored on
four different splits:

In [2]:
X = network[FEATURES]
y = network["jan_mean"]

for seed in range(4):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=seed)
    print("seed", seed, round(LinearRegression().fit(X_train, y_train).score(X_test, y_test), 3))

seed 0 0.888
seed 1 0.875
seed 2 0.876
seed 3 0.873


The four scores range from 0.873 to 0.888, and the only thing that changed was which stations were
in the test quarter.

### Folds instead of a single split

`cross_val_score` takes an unfitted model, the whole table and the number of folds, and returns one
score for each fold:

In [3]:
scores = cross_val_score(LinearRegression(), X, y, cv=5)

print(scores.round(3))
print("mean", scores.mean().round(3), "| spread (standard deviation)", scores.std().round(3))

[0.874 0.881 0.889 0.881 0.875]
mean 0.88 | spread (standard deviation) 0.005


Five fits, each on 960 stations and scored on the other 240. The mean, 0.88, uses every station once
as a test, and the spread, 0.005, says the score barely depends on which fifth is held out. Report
both, since a mean with no spread hides how much it could move. This notebook cross-validates all
1,200 stations to keep the examples short. When a test set is kept for the end, split it off first
and make the folds from the training stations alone.

### The folds themselves

A splitter object makes the folds, and `split` yields the positions of each fold's training and test
rows. `shuffle=True` mixes the rows first, and `random_state` fixes the mix:

In [4]:
folds = KFold(5, shuffle=True, random_state=0)

for number, (train, test) in enumerate(folds.split(X)):
    print(f"fold {number}: {len(train)} to train, {len(test)} to test, first test rows {test[:4]}")

fold 0: 960 to train, 240 to test, first test rows [ 1  5  8 10]
fold 1: 960 to train, 240 to test, first test rows [ 2 15 19 30]
fold 2: 960 to train, 240 to test, first test rows [ 0  6  9 12]
fold 3: 960 to train, 240 to test, first test rows [ 3  4  7 13]
fold 4: 960 to train, 240 to test, first test rows [11 23 24 25]


Each test fold holds 240 stations taken from all over the table. Passing the splitter as `cv` scores
the model on exactly these folds:

In [5]:
shuffled = cross_val_score(LinearRegression(), X, y, cv=folds)

print(shuffled.round(3))
print("mean", shuffled.mean().round(3), "| spread", shuffled.std().round(3))

[0.897 0.869 0.865 0.874 0.89 ]
mean 0.879 | spread 0.013


The mean hardly moved, from 0.88 to 0.879, because this table's rows come in no particular order.
The individual folds did change, and the spread with them. The first Common error shows a table
whose order does matter.

### Comparing models on the same folds

Comparing models fairly means scoring them on the same folds, so that each comparison is between two
models tested on the same stations:

In [6]:
linear = cross_val_score(LinearRegression(), X, y, cv=folds)
forest = cross_val_score(RandomForestRegressor(random_state=0), X, y, cv=folds)

print("linear", linear.round(3), "mean", linear.mean().round(3), "spread", linear.std().round(3))
print("forest", forest.round(3), "mean", forest.mean().round(3), "spread", forest.std().round(3))
print("forest minus linear", (forest - linear).round(3))
print("mean difference", (forest - linear).mean().round(3))

linear [0.897 0.869 0.865 0.874 0.89 ] mean 0.879 spread 0.013
forest [0.901 0.878 0.87  0.885 0.888] mean 0.885 spread 0.01
forest minus linear [ 0.004  0.009  0.006  0.011 -0.002]
mean difference 0.005


The forest scored higher in four of the five folds, by at most 0.011, and lower in one. Both models
scored highest on the first fold and lowest on the third, so much of each model's spread comes from
how hard a fold's stations are, which the two share. The differences on the last lines remove that,
and they average 0.005. The forest is slightly ahead on these stations, and five folds whose
training rows overlap are too few to say more.

### Stratified folds for a rare class

Frost in June happens at 59 of the 1,200 stations. Shuffled folds that ignore the answer can give
some folds far more frost stations than others. The cell counts the frost stations in each test
fold:

In [7]:
frost = network["june_frost"]

plain = KFold(5, shuffle=True, random_state=0)
stratified = StratifiedKFold(5, shuffle=True, random_state=0)

print("KFold:          ", [int(frost.iloc[test].sum()) for _, test in plain.split(X)])
print("StratifiedKFold:", [int(frost.iloc[test].sum()) for _, test in stratified.split(X, frost)])

KFold:           [16, 16, 7, 5, 15]
StratifiedKFold: [11, 12, 12, 12, 12]


With `KFold`, one fold tested on 16 frost stations and another on 5. `StratifiedKFold` takes the
answer as well as the rows and deals the frost stations out evenly, 11 or 12 to a fold, so every
fold's score is about the same kind of problem. The **Imbalanced Classes** notebook relies on this.

### Training scores beside test scores

`cross_validate` returns a dictionary of arrays. It also takes a list of scoring names, as the first
Common error shows, and `return_train_score=True` adds each fold's score on its own training rows.
Here the snow model, scaled in a pipeline, is scored on accuracy, the share of right answers, on
both the rows it trained on and the rows it did not:

In [8]:
snow = network["snow_on_may_1"]
snow_model = make_pipeline(StandardScaler(), LogisticRegression())

result = cross_validate(snow_model, X, snow, cv=stratified, scoring="accuracy",
                        return_train_score=True)

print(sorted(result))
print("train:", result["train_score"].round(3), "mean", result["train_score"].mean().round(3))
print("test: ", result["test_score"].round(3), "mean", result["test_score"].mean().round(3))

['fit_time', 'score_time', 'test_score', 'train_score']
train: [0.849 0.861 0.854 0.855 0.849] mean 0.854
test:  [0.875 0.817 0.858 0.842 0.867] mean 0.852


The keys include `fit_time` and `score_time`, which differ on every run and are not printed. The
model is right on 0.854 of its training stations and 0.852 of its test stations. Scores this close
mean it is not memorizing its training rows, and a large gap would be the first sign that it was, as
the **Train and Test** notebook showed with a tree.

## Your turn

Four tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/06-cross-validation-solutions.ipynb).

**1.** Cross-validate `LinearRegression` on `X` and `y` with `KFold(10, shuffle=True, random_state=0)`, and print the ten scores, their mean and their spread, rounded to three places.

In [9]:
# your code here

**2.** Count the snow stations in each test fold of `KFold(5, shuffle=True, random_state=0)` and of `StratifiedKFold(5, shuffle=True, random_state=0)`, and print both lists.

In [10]:
# your code here

**3.** Score `RandomForestRegressor(random_state=0)` and `LinearRegression` with `scoring="neg_mean_absolute_error"` on the same shuffled five folds, and print each model's mean error in degrees as a positive number.

In [11]:
# your code here

**4.** Sort `network` by `elevation_m`, then cross-validate `LinearRegression` on it with `cv=5` and with `KFold(5, shuffle=True, random_state=0)`, and print both sets of scores.

In [12]:
# your code here

## Common errors

### No error, and R² below zero from folds of a sorted table

A file exported sorted by its own target is common, since sorting is how people read a table. Here
are the stations sorted by January mean, scored with `cv=5`:

In [13]:
by_january = network.sort_values("jan_mean").reset_index(drop=True)
X_sorted, y_sorted = by_january[FEATURES], by_january["jan_mean"]

in_order = cross_validate(LinearRegression(), X_sorted, y_sorted, cv=5,
                          scoring=["r2", "neg_mean_absolute_error"])

print("R²:                ", in_order["test_r2"].round(3))
print("error in degrees:  ", (-in_order["test_neg_mean_absolute_error"]).round(3))
print("spread of each fold's January means:",
      [round(y_sorted.iloc[test].std(), 2) for _, test in KFold(5).split(X_sorted)])

residual_means = [
    (y_sorted.iloc[test] - LinearRegression().fit(X_sorted.iloc[train], y_sorted.iloc[train])
     .predict(X_sorted.iloc[test])).mean()
    for train, test in KFold(5).split(X_sorted)
]
print("mean residual in each fold:", np.round(residual_means, 2))

R²:                 [ 0.099 -1.847 -4.638 -2.76  -0.886]
error in degrees:   [1.15  0.769 0.769 0.834 1.258]
spread of each fold's January means: [1.44, 0.58, 0.4, 0.53, 1.08]
mean residual in each fold: [-0.77 -0.28  0.01  0.02  1.21]


`cv=5` split the rows in order, so each test fold held one band of January means, from the coldest
fifth to the mildest. The three middle folds' errors, 0.769 to 0.834 degrees, are about the size of
a shuffled fold's, which the next Common error prints. Their January means vary little, with
standard deviations of 0.58, 0.4 and 0.53 degrees, and R² measures error against the variation of
the rows being scored, so with errors that small it still fell as low as -4.638. The first and last
folds had larger errors, 1.15 and 1.258 degrees, and most of them went one way. The coldest fifth
came out 0.77 degrees too warm on average and the mildest fifth 1.21 degrees too cold. Sorting by
the measured January sent the stations that are colder or milder than their features suggest to the
two ends, and a model that predicts from the features misses those stations in one direction.
Shuffle, and the scores come back:

In [14]:
mixed = KFold(5, shuffle=True, random_state=0)

print(cross_val_score(LinearRegression(), X_sorted, y_sorted, cv=mixed).round(3))

[0.883 0.896 0.875 0.872 0.87 ]


### No error, and negative mean absolute errors

In [15]:
errors = cross_val_score(LinearRegression(), X, y, cv=folds, scoring="neg_mean_absolute_error")

print(errors.round(3))

[-0.753 -0.852 -0.84  -0.814 -0.813]


An error cannot be negative, and these are. scikit-learn's scores all follow the rule that larger is
better, so a scoring name that starts with `neg_` returns the error with its sign flipped, and
-0.753 means the model was 0.753 degrees off on average in that fold. Negate the scores to read
them, as `-errors`, and remember when choosing the best that the largest, nearest zero, is the
smallest error.

### UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.

In [16]:
first_forty = network.head(40)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    frost_model = make_pipeline(StandardScaler(), LogisticRegression())
    scores = cross_val_score(frost_model, first_forty[FEATURES], first_forty["june_frost"],
                             cv=StratifiedKFold(5))

print("frost stations among the 40:", int(first_forty["june_frost"].sum()))
print(scores.round(3))
for warning in caught:
    if warning.category is not DeprecationWarning:
        print(f"{warning.category.__name__}: {warning.message}")

frost stations among the 40: 2
[0.875 1.    1.    0.875 0.875]


Forty stations held only two with a frost in June, and two stations can be in at most two of the
five test folds, so at least three folds tested on no frost station at all, and their scores say
nothing about finding frost. The warning names the problem, and the cure is more rows of the rare
class, or no more folds than it has members. The loop leaves out a `DeprecationWarning` that SciPy
raises inside every fit of the logistic regression, which has nothing to do with the folds.

## Recap

- Score a model on every row by turns with `cross_val_score`, and report the mean and the spread
- Shuffle the folds with a seeded `KFold`, and compare models on the same folds
- Keep a rare class evenly spread across folds with `StratifiedKFold`
- Get several scores, and training scores, from `cross_validate`, and read `neg_` scores as errors

## What is next

The **Regression Metrics** notebook looks at what R² and the other single numbers for a regressor
measure, and at what each of them leaves out.

---

&#8592; **Previous:** [Pipelines](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/05-pipelines.ipynb)  &nbsp;·&nbsp;  [scikit-learn, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Regression Metrics](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/07-regression-metrics.ipynb) &#8594;
